# MNIST High Accuracy Challenge

Objetivo: Alcanzar >99.4% de accuracy en MNIST usando solo redes fully-connected (MLP).

Técnicas utilizadas:
- Batch Normalization
- Learning Rate Scheduling
- Data Augmentation
- Dropout
- Arquitectura optimizada

## Imports

In [1]:
import torch
import torchvision
import torch.nn as nn
from tqdm import tqdm
import multiprocessing
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np

print("Torch version:", torch.__version__)

# Set random seed for reproducibility
SEED = 777
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch version: 2.9.0+cu126
Device: cuda


## Data Augmentation Configuration

In [2]:
# Para MNIST no normalizamos - ToTensor ya escala a [0,1] y es suficiente
# La normalización adicional puede ser contraproducente para este dataset

train_transform = transforms.Compose([
    # Data Augmentation AGRESIVO para mayor robustez
    transforms.RandomAffine(
        degrees=20,              # Rotación hasta 20 grados
        translate=(0.15, 0.15),  # Traslación hasta 15%
        scale=(0.85, 1.15),      # Escala entre 85% y 115%
        shear=15                 # Shear hasta 15 grados
    ),
    transforms.ToTensor(),       # Convierte a tensor y escala a [0,1]
    transforms.RandomErasing(
        p=0.2,                   # 20% de probabilidad
        scale=(0.02, 0.25)       # Borra entre 2% y 25% de la imagen
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor()        # Solo convertir a tensor para test
])

Calculating MNIST mean and std...
Calculated Mean: (0.13066047430038452,), Std: (0.30810782313346863,)
Calculated Mean: (0.13066047430038452,), Std: (0.30810782313346863,)


## Dataset Class

In [3]:
class MNIST_dataset(Dataset):
    
    def __init__(self, partition="train", transform=None):
        print("\nLoading MNIST ", partition, " Dataset...")
        self.partition = partition
        self.transform = transform
        
        if self.partition == "train":
            self.data = torchvision.datasets.MNIST('.data/', train=True, download=True)
        else:
            self.data = torchvision.datasets.MNIST('.data/', train=False, download=True)
        
        print("\tTotal Len.: ", len(self.data), "\n", 50*"-")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx][0]
        image = self.transform(image)
        image = image.view(-1)

        label = self.data[idx][1]
        # Devolvemos el índice de la clase (Long) en lugar de One-Hot
        # Esto es necesario para usar label_smoothing en CrossEntropyLoss de forma eficiente
        label = torch.tensor(label, dtype=torch.long)

        return {"idx": idx, "img": image, "label": label}

## Neural Network with Batch Normalization and Dropout

In [4]:
class Net(nn.Module):
    def __init__(self, sizes=[[784, 1024], [1024, 1024], [1024, 1024], [1024, 512], [512, 10]], 
                 dropout_rate=0.3, criterion=None):
        super(Net, self).__init__()
        
        self.layers = nn.ModuleList()
        
        for i in range(len(sizes) - 1):
            dims = sizes[i]
            self.layers.append(nn.Linear(dims[0], dims[1]))
            self.layers.append(nn.BatchNorm1d(dims[1]))
            self.layers.append(nn.GELU()) # GELU suele funcionar mejor que ReLU en redes profundas
            self.layers.append(nn.Dropout(dropout_rate))
        
        dims = sizes[-1]
        self.classifier = nn.Linear(dims[0], dims[1])
        self.criterion = criterion
        
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # He initialization (Kaiming)
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x, y=None):
        for layer in self.layers:
            x = layer(x)
        x = self.classifier(x)
        
        if y is not None:
            loss = self.criterion(x, y)
            return loss, x
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Load Data and Create DataLoaders

In [5]:
train_dataset = MNIST_dataset(partition="train", transform=train_transform)
test_dataset = MNIST_dataset(partition="test", transform=test_transform)

# Aumentamos el batch_size para acelerar el entrenamiento en GPU
batch_size = 512 

# Configuración segura de workers para Cluster/Compartido
# Intentamos leer de variables de entorno comunes en clusters (SLURM)
import os
if 'SLURM_CPUS_PER_TASK' in os.environ:
    num_workers = int(os.environ['SLURM_CPUS_PER_TASK'])
else:
    # Si no estamos en un job de SLURM, limitamos a 4 para no saturar el nodo de login
    num_workers = min(4, multiprocessing.cpu_count())

print("Num workers configured:", num_workers)

# pin_memory=True acelera la transferencia Host-to-Device
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)


Loading MNIST  train  Dataset...
	Total Len.:  60000 
 --------------------------------------------------

Loading MNIST  test  Dataset...
	Total Len.:  10000 
 --------------------------------------------------
Num workers 2


## Initialize Model and Training Configuration

In [6]:
# Reducimos Label Smoothing a 0.05 para permitir mayor confianza en las predicciones
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

num_classes = 10
# Arquitectura EXTRA ancha para maximizar capacidad
net = Net(
    sizes=[
        [784, 2048],      # Aumentado de 1500 a 2048
        [2048, 2048],     # Aumentado de 1500 a 2048
        [2048, 1024],     # Aumentado de 1000 a 1024
        [1024, 512], 
        [512, num_classes]
    ], 
    dropout_rate=0.25,    # Reducido de 0.2 a 0.25 para más regularización con red más grande
    criterion=criterion
)

print(net)
print("Params: ", count_parameters(net))

# Ajustamos LR inicial - usamos AdamW que es más estable que SGD
optimizer = optim.AdamW(net.parameters(), lr=0.001, weight_decay=0.01)

# Usamos CosineAnnealingWarmRestarts para mejor convergencia
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=10,        # Reinicia cada 10 epochs
    T_mult=2,      # Duplica el periodo tras cada reinicio
    eta_min=1e-6
)

net = net.to(device)
epochs = 200

Net(
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=1500, bias=True)
    (1): BatchNorm1d(1500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=1500, out_features=1500, bias=True)
    (5): BatchNorm1d(1500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=1500, out_features=1000, bias=True)
    (9): BatchNorm1d(1000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): GELU(approximate='none')
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=1000, out_features=500, bias=True)
    (13): BatchNorm1d(500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): GELU(approximate='none')
    (15): Dropout(p=0.2, inplace=False)
  )
  (classifier): Linear(in_features=500, out_features

## Training Loop

In [7]:
print("\n---- Start Training ----")
best_accuracy = -1
best_epoch = 0

# Inicializamos GradScaler para Mixed Precision Training (AMP)
scaler = torch.amp.GradScaler('cuda')

for epoch in range(epochs):
    
    # TRAIN NETWORK
    train_loss, train_correct = 0, 0
    net.train()
    
    for batch in train_dataloader:
        images = batch["img"].to(device)
        labels = batch["label"].to(device)
        ids = batch["idx"].to('cpu').numpy()
        
        optimizer.zero_grad()
        
        # Usamos autocast para Mixed Precision
        with torch.amp.autocast('cuda'):
            loss, outputs = net(images, labels)
        
        # Escalamos la pérdida y hacemos backward
        scaler.scale(loss).backward()
        
        # Gradient clipping para estabilidad
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        pred = torch.argmax(outputs, dim=1)
        train_correct += pred.eq(labels).sum().item()
        train_loss += loss.item()

    train_loss /= len(train_dataloader) 
    train_accuracy = 100. * train_correct / len(train_dataloader.dataset)
    
    # TEST NETWORK
    test_loss, test_correct = 0, 0
    net.eval()
    
    with torch.no_grad():
        for batch in test_dataloader:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)
            ids = batch["idx"].to('cpu').numpy()
            
            outputs = net(images)
            test_loss += criterion(outputs, labels).item()
            
            pred = torch.argmax(outputs, dim=1)
            test_correct += pred.eq(labels).sum().item()
    
    test_loss /= len(test_dataloader)
    test_accuracy = 100. * test_correct / len(test_dataloader.dataset)
    
    # Actualizamos el scheduler cada epoch (CosineAnnealing)
    scheduler.step()
    
    print("[Epoch {:3d}] Train: {:.2f}% | Test: {:.2f}% | Loss: {:.4f} | LR: {:.6f}".format(
        epoch + 1, train_accuracy, test_accuracy, test_loss, optimizer.param_groups[0]['lr']
    ))
    
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        best_epoch = epoch
        torch.save(net.state_dict(), "best_model_high_acc.pt")
        print(f"  ★ New best: {best_accuracy:.2f}%")

print("\n" + "="*50)
print(f"BEST TEST ACCURACY: {best_accuracy:.2f}% in epoch {best_epoch+1}")
print("="*50)


---- Start Training ----
[Epoch  1] Train: 75.16% | Test: 95.53% | Loss: 0.4755 | LR: 0.20000
[Epoch  1] Train: 75.16% | Test: 95.53% | Loss: 0.4755 | LR: 0.20000
[Epoch  2] Train: 87.16% | Test: 96.42% | Loss: 0.4422 | LR: 0.20000
[Epoch  2] Train: 87.16% | Test: 96.42% | Loss: 0.4422 | LR: 0.20000
[Epoch  3] Train: 89.69% | Test: 97.02% | Loss: 0.4070 | LR: 0.20000
[Epoch  3] Train: 89.69% | Test: 97.02% | Loss: 0.4070 | LR: 0.20000
[Epoch  4] Train: 91.19% | Test: 97.48% | Loss: 0.3910 | LR: 0.20000
[Epoch  4] Train: 91.19% | Test: 97.48% | Loss: 0.3910 | LR: 0.20000
[Epoch  5] Train: 92.10% | Test: 97.85% | Loss: 0.3773 | LR: 0.20000
[Epoch  5] Train: 92.10% | Test: 97.85% | Loss: 0.3773 | LR: 0.20000
[Epoch  6] Train: 92.70% | Test: 97.94% | Loss: 0.3693 | LR: 0.20000
[Epoch  6] Train: 92.70% | Test: 97.94% | Loss: 0.3693 | LR: 0.20000
[Epoch  7] Train: 93.24% | Test: 97.99% | Loss: 0.3639 | LR: 0.20000
[Epoch  7] Train: 93.24% | Test: 97.99% | Loss: 0.3639 | LR: 0.20000
[Epoch  

## Load Best Model and Final Evaluation

In [8]:
net.load_state_dict(torch.load("best_model_high_acc.pt"))

test_loss, test_correct = 0, 0
net.eval()

with torch.no_grad():
    with tqdm(iter(test_dataloader), desc="Test " + str(epoch), unit="batch") as tepoch:
        for batch in tepoch:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)
            ids = batch["idx"].to('cpu').numpy()
            
            outputs = net(images)
            test_loss += criterion(outputs, labels).item()
            
            # labels ya son índices
            # labels = torch.argmax(labels, dim=1)
            pred = torch.argmax(outputs, dim=1)
            test_correct += pred.eq(labels).sum().item()

test_loss /= len(test_dataloader)
test_accuracy = 100. * test_correct / len(test_dataloader.dataset)
print(f"Final best acc: {test_accuracy:.2f}")

Test 59: 100%|██████████| 20/20 [00:02<00:00,  9.87batch/s]

Final best acc: 99.28
